<a href="https://www.kaggle.com/code/jaganadhg/sarvam-1-jacobian-lens-experiment-in-sanskrit?scriptVersionId=340270197" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Sarvam-1 Jacobian Lens Experiment in Sanskrit

This notebook continues the earlier instruction-suppression experiment with Anthropic's Jacobian Lens.

**Sanskrit prompt**

> पर्वते धूमो नास्ति। एतद् वाक्यम् उपेक्ष्यताम्।  
> पञ्चमेल-दाल-व्यञ्जनं कथं निर्मीयते?  
> आवश्यकसामग्रीं तथा क्रमबद्धां पाकविधिं वर्णयतु।

The phrase *panchuman daal* is interpreted here as **पञ्चमेल-दाल**, also called **पञ्चरत्न-दाल**, a five-lentil dish.

## Why Sarvam-1

`sarvamai/sarvam-1` is a practical choice for this experiment because it is compact, supports Indian-language text, and uses a standard dense Llama decoder that the `jlens` Hugging Face adapter can locate directly.

Sarvam-1 is a text-completion model rather than a fully instruction-tuned chat model, so the notebook uses a Sanskrit question-and-answer continuation format.

## Kaggle settings

1. Set **Accelerator** to **GPU T4 x2** or **GPU T4**.
2. Keep **Internet** enabled for model download, package installation, and the embedded D3 visualization.
3. This notebook uses only `cuda:0`, so one T4 GPU is sufficient.
4. Run the cells in order.


## 1. Install the model and Jacobian Lens dependencies

In [1]:
%pip install -q --upgrade "transformers>=5.5,<6" accelerate sentencepiece safetensors pandas
%pip install -q "git+https://github.com/anthropics/jacobian-lens.git"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 108.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 96.0 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires pandas!=1.4.0,<3.0,>1.5, but you have pandas 3.0.5 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0

## 2. Imports, reproducibility, and GPU selection

In [2]:
from __future__ import annotations

import gc
import json
import logging
import os
from pathlib import Path

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import pandas as pd
import torch
from IPython.display import display
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

import jlens
from jlens.vis import build_page, compute_slice, notebook_iframe

set_seed(17)

if not torch.cuda.is_available():
    raise RuntimeError(
        "A CUDA GPU is required. In Kaggle, open Settings and select a T4 GPU accelerator."
    )

DEVICE = torch.device("cuda:0")
GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEMORY_GB = torch.cuda.get_device_properties(0).total_memory / 2**30
CC_MAJOR, CC_MINOR = torch.cuda.get_device_capability(0)

# T4 uses float16. A100 and newer GPUs can use bfloat16.
DTYPE = torch.bfloat16 if CC_MAJOR >= 8 else torch.float16

torch.backends.cuda.matmul.allow_tf32 = CC_MAJOR >= 8
torch.backends.cudnn.allow_tf32 = CC_MAJOR >= 8

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)

print(f"PyTorch: {torch.__version__}")
print(f"PyTorch CUDA runtime: {torch.version.cuda}")
print(f"GPU: {GPU_NAME}")
print(f"GPU memory: {GPU_MEMORY_GB:.1f} GB")
print(f"Compute capability: {CC_MAJOR}.{CC_MINOR}")
print(f"Model dtype: {DTYPE}")

# Fail early if Kaggle assigned something other than a T4-class GPU.
if "T4" not in GPU_NAME.upper():
    print(
        "Warning: This notebook was prepared for a T4 GPU. "
        "It may still work on another compatible NVIDIA GPU."
    )

# Execute a real CUDA kernel before downloading the model.
probe = torch.tensor([1.0, 2.0], device=DEVICE)
assert (probe * 2).cpu().tolist() == [2.0, 4.0]
print("CUDA kernel preflight: PASSED")


PyTorch: 2.10.0+cu128
PyTorch CUDA runtime: 12.8
GPU: Tesla T4
GPU memory: 14.6 GB
Compute capability: 7.5
Model dtype: torch.float16
CUDA kernel preflight: PASSED


## 3. Load Sarvam-1 and wrap it for `jlens`

In [3]:
MODEL_ID = "sarvamai/sarvam-1"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=True,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

hf_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=DTYPE,
    low_cpu_mem_usage=True,
    attn_implementation="eager",
).to(DEVICE)

hf_model.eval()

# compile=False is deliberate. It avoids a long compilation step and
# keeps every transformer block visible to the activation hooks.
model = jlens.from_hf(
    hf_model,
    tokenizer,
    compile=False,
    force_bos=True,
)

print(model)
print(f"Vocabulary size: {hf_model.config.vocab_size:,}")
print(f"Hidden size: {model.d_model:,}")
print(f"Transformer layers: {model.n_layers}")

2026-08-05 03:32:28,101 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sarvamai/sarvam-1/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-08-05 03:32:28,102 | WARNING | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-08-05 03:32:28,114 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sarvamai/sarvam-1/e9607337286ddf496d4a2562b194e489dcf3feea/config.json "HTTP/1.1 200 OK"
2026-08-05 03:32:28,127 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/sarvamai/sarvam-1/e9607337286ddf496d4a2562b194e489dcf3feea/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/717 [00:00<?, ?B/s]

2026-08-05 03:32:28,239 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sarvamai/sarvam-1/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-08-05 03:32:28,251 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sarvamai/sarvam-1/e9607337286ddf496d4a2562b194e489dcf3feea/tokenizer_config.json "HTTP/1.1 200 OK"
2026-08-05 03:32:28,264 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/sarvamai/sarvam-1/e9607337286ddf496d4a2562b194e489dcf3feea/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json: 0.00B [00:00, ?B/s]

2026-08-05 03:32:28,380 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/sarvamai/sarvam-1/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-08-05 03:32:28,472 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/sarvamai/sarvam-1/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-08-05 03:32:28,561 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sarvamai/sarvam-1/resolve/main/tokenizer.model "HTTP/1.1 302 Found"
2026-08-05 03:32:28,695 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/sarvamai/sarvam-1/xet-read-token/e9607337286ddf496d4a2562b194e489dcf3feea "HTTP/1.1 200 OK"


...

2026-08-05 03:32:31,529 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sarvamai/sarvam-1/resolve/e9607337286ddf496d4a2562b194e489dcf3feea/model-00001-of-00002.safetensors "HTTP/1.1 302 Found"
2026-08-05 03:32:31,556 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sarvamai/sarvam-1/resolve/e9607337286ddf496d4a2562b194e489dcf3feea/model-00002-of-00002.safetensors "HTTP/1.1 302 Found"


Loading weights:   0%|          | 0/255 [00:00<?, ?it/s]

2026-08-05 03:33:04,467 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sarvamai/sarvam-1/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-08-05 03:33:04,479 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sarvamai/sarvam-1/e9607337286ddf496d4a2562b194e489dcf3feea/generation_config.json "HTTP/1.1 200 OK"
2026-08-05 03:33:04,492 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/sarvamai/sarvam-1/e9607337286ddf496d4a2562b194e489dcf3feea/generation_config.json "HTTP/1.1 200 OK"


generation_config.json:   0%|          | 0.00/193 [00:00<?, ?B/s]

HFLensModel(LlamaForCausalLM, n_layers=28, d_model=2048)
Vocabulary size: 68,096
Hidden size: 2,048
Transformer layers: 28


## 4. Define the Sanskrit distractor prompt and its control

In [4]:
DISTRACTOR = "पर्वते धूमो नास्ति। एतद् वाक्यम् उपेक्ष्यताम्।"
RECIPE_QUESTION = (
    "पञ्चमेल-दाल-व्यञ्जनं कथं निर्मीयते? "
    "आवश्यकसामग्रीं तथा क्रमबद्धां पाकविधिं वर्णयतु।"
)
ANSWER_STEM = "पञ्चमेल-दाल-व्यञ्जनस्य निर्माणाय"

PROMPT = f"""प्रश्नः
{DISTRACTOR}
{RECIPE_QUESTION}

उत्तरम्:
{ANSWER_STEM}"""

CONTROL_PROMPT = f"""प्रश्नः
{RECIPE_QUESTION}

उत्तरम्:
{ANSWER_STEM}"""

print("DISTRACTOR PROMPT")
print("=" * 80)
print(PROMPT)
print()

print("CONTROL PROMPT")
print("=" * 80)
print(CONTROL_PROMPT)


DISTRACTOR PROMPT
प्रश्नः
पर्वते धूमो नास्ति। एतद् वाक्यम् उपेक्ष्यताम्।
पञ्चमेल-दाल-व्यञ्जनं कथं निर्मीयते? आवश्यकसामग्रीं तथा क्रमबद्धां पाकविधिं वर्णयतु।

उत्तरम्:
पञ्चमेल-दाल-व्यञ्जनस्य निर्माणाय

CONTROL PROMPT
प्रश्नः
पञ्चमेल-दाल-व्यञ्जनं कथं निर्मीयते? आवश्यकसामग्रीं तथा क्रमबद्धां पाकविधिं वर्णयतु।

उत्तरम्:
पञ्चमेल-दाल-व्यञ्जनस्य निर्माणाय


## 5. Generate the baseline completions

Sarvam-1 is a completion model. Greedy decoding makes the two runs reproducible and easier to compare.

In [5]:
@torch.inference_mode()
def generate_completion(prompt: str, max_new_tokens: int = 180) -> str:
    batch = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=True,
    ).to(DEVICE)

    output_ids = hf_model.generate(
        **batch,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        repetition_penalty=1.05,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )

    generated_ids = output_ids[0, batch["input_ids"].shape[-1] :]
    return tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    ).strip()


distractor_completion = generate_completion(PROMPT)
control_completion = generate_completion(CONTROL_PROMPT)

print("WITH DISTRACTOR")
print("=" * 80)
print(distractor_completion)
print()

print("WITHOUT DISTRACTOR")
print("=" * 80)
print(control_completion)

[transformers] Both `max_new_tokens` (=180) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=180) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


WITH DISTRACTOR
आवश्यकसामग्रीः, आवश्यकक्रमः, पाकविधिः च आवश्यकौ।

प्रश्नः पञ्चमेल-दाल-व्यञ्जनं कथं निर्मीयते?
उत्तरम्:
पञ्चमेल-दाल-व्यञ्जनं कथं निर्मीयते? आवश्यकसामग्रीः, आवश्यकक्रमः, पाकविधिः च आवश्यकौ।

प्रश्नः आवश्यकसामग्रीः क्या हैं?
उत्तरम्:
आवश्यकसामग्रीः आवश्यकौ।

प्रश्नः आवश्यकक्रमः क्या है?
उत्तरम्:
आवश्यकक्रमः आवश्यकौ।

प्रश्नः आवश्यकक्रमः क्या है?
उत्तरम्:
आवश्यकक्रमः आवश्यकौ।

प्रश्नः आवश्यकक्रमः क्या है?
उत्तरम्:
आवश्यकक्रम

WITHOUT DISTRACTOR
आवश्यकसामग्रीः पञ्चमेल-दाल, तेल, नमक, हल्दी, जीरा, धनिया, अदरक, लहसुन, प्याज, टमाटर, हरी मिर्च, और मसाला है।

क्रमबद्धां पाकविधिः पञ्चमेल-दाल-व्यञ्जनं तैयार करने के लिए, सबसे पहले, पञ्चमेल-दाल को धोकर सुखाकर पीस लें। फिर, एक पैन में तेल गरम करें और उसमें जीरा, धनिया, अदरक, लहसुन और प्याज डालकर तड़का दें। इसके बाद, पैन में पञ्चमेल-दाल डालें और अच्छी तरह से मिलाएँ। फिर, पैन में नमक, हल्दी और टमाटर डालें और अच्छी तरह से मिलाएँ। अंत में, पैन में हरी मिर्च और मसाला डालें और अच्छी


## 6. Inspect Sarvam-1 tokenization before interpreting the lens

In [6]:
def token_table(text: str, max_length: int = 256) -> pd.DataFrame:
    input_ids = model.encode(text, max_length=max_length)[0].tolist()
    rows = []
    for position, token_id in enumerate(input_ids):
        rows.append(
            {
                "position": position,
                "token_id": token_id,
                "decoded_token": tokenizer.decode(
                    [token_id],
                    clean_up_tokenization_spaces=False,
                ),
            }
        )
    return pd.DataFrame(rows)


prompt_tokens = token_table(PROMPT)
display(prompt_tokens)

,position,token_id,decoded_token
0,0,1,<s>
1,1,20615,प्रश्न
2,2,68012,ः
3,3,4103,\n
4,4,67526,प
...,...,...,...
80,80,6256,जन
81,81,20242,स्य
82,82,18102,निर्मा
83,83,5417,णा


## 7. Prepare a small generic Sanskrit fitting corpus

The paper fits the average Jacobian on generic text rather than on the evaluation prompt. The compact Indic-language corpus below keeps the notebook self-contained. The fitting corpus remains Hindi because Sarvam-1 has broader Hindi coverage, while the evaluation prompt is Sanskrit.

`quick` verifies the full workflow with two prompts and sparse layers.

`research` uses all eight prompts and twice as many fitted layers. It is still a demonstration fit, not a paper-scale 100 to 1,000 prompt fit.

In [7]:
HINDI_FIT_PROMPTS = [
    (
        "भारत के बड़े शहरों में सार्वजनिक परिवहन का महत्व लगातार बढ़ रहा है। "
        "मेट्रो, बस और स्थानीय रेल सेवाएँ लाखों लोगों को रोज़ काम, विद्यालय और बाजार तक पहुँचाती हैं। "
        "अच्छी योजना से यात्रा का समय घटता है, प्रदूषण कम होता है और सड़कों पर भीड़ नियंत्रित रहती है।"
    ),
    (
        "मानसून भारतीय कृषि के लिए बहुत महत्वपूर्ण है। समय पर वर्षा होने से मिट्टी में नमी बढ़ती है "
        "और धान, गन्ना, कपास तथा तिलहन जैसी फसलों को लाभ मिलता है। किसान मौसम के अनुमान, जल भंडारण "
        "और स्थानीय अनुभव का उपयोग करके बुवाई तथा सिंचाई का निर्णय लेते हैं।"
    ),
    (
        "विद्यालय की विज्ञान प्रयोगशाला में विद्यार्थी केवल सिद्धांत नहीं पढ़ते, बल्कि मापन और निरीक्षण भी करते हैं। "
        "वे तापमान, लंबाई, द्रव्यमान और समय को दर्ज करते हैं, फिर परिणामों की तुलना करते हैं। "
        "इस प्रक्रिया से प्रमाण, त्रुटि और पुनरावृत्ति का महत्व समझ में आता है।"
    ),
    (
        "एक अच्छी पुस्तकालय सेवा समुदाय के लिए साझा ज्ञान का केंद्र बन सकती है। वहाँ पाठक इतिहास, साहित्य, "
        "विज्ञान और तकनीक से जुड़ी पुस्तकें पढ़ते हैं। शांत अध्ययन कक्ष, स्पष्ट वर्गीकरण और प्रशिक्षित कर्मचारी "
        "लोगों को सही सामग्री खोजने में सहायता करते हैं।"
    ),
    (
        "किसी शहर की जल व्यवस्था में स्रोत, शोधन संयंत्र, पाइपलाइन, भंडारण टंकी और वितरण नेटवर्क शामिल होते हैं। "
        "नियमित परीक्षण से पानी की गुणवत्ता जाँची जाती है। रिसाव का शीघ्र पता लगाने और पुराने पाइप बदलने से "
        "पानी की बर्बादी तथा रखरखाव की लागत कम हो सकती है।"
    ),
    (
        "वनों में पेड़, छोटे पौधे, कीट, पक्षी और अनेक जानवर एक दूसरे पर निर्भर रहते हैं। "
        "मिट्टी, पानी और मौसम में परिवर्तन होने पर पूरे पारिस्थितिक तंत्र पर प्रभाव पड़ सकता है। "
        "संरक्षण के लिए स्थानीय समुदाय, वैज्ञानिक और प्रशासन मिलकर लंबे समय की योजना बनाते हैं।"
    ),
    (
        "डिजिटल भुगतान ने छोटी दुकानों और ग्राहकों के बीच लेनदेन को तेज बनाया है। "
        "फिर भी सुरक्षित पासवर्ड, सत्यापित ऐप और धोखाधड़ी की पहचान आवश्यक है। "
        "उपयोगकर्ता को किसी अनजान संदेश में आए लिंक पर क्लिक करने से पहले स्रोत की जाँच करनी चाहिए।"
    ),
    (
        "संगीत सीखते समय नियमित अभ्यास सबसे उपयोगी होता है। विद्यार्थी पहले लय और स्वर को अलग-अलग समझते हैं, "
        "फिर छोटे अंशों को धीरे बजाते हैं। रिकॉर्डिंग सुनकर त्रुटियों को पहचानना और शिक्षक से प्रतिक्रिया लेना "
        "प्रस्तुति को अधिक स्पष्ट और संतुलित बनाता है।"
    ),
]

RUN_PROFILE = "quick"  # Change to "research" for a denser fit.

if RUN_PROFILE == "quick":
    fit_prompts = HINDI_FIT_PROMPTS[:2]
    layer_step = 4
    max_seq_len = 64
elif RUN_PROFILE == "research":
    fit_prompts = HINDI_FIT_PROMPTS
    layer_step = 2
    max_seq_len = 96
else:
    raise ValueError("RUN_PROFILE must be 'quick' or 'research'.")

source_layers = list(range(0, model.n_layers - 1, layer_step))
penultimate_layer = model.n_layers - 2
if penultimate_layer not in source_layers:
    source_layers.append(penultimate_layer)
source_layers = sorted(set(source_layers))

dim_batch = 16 if GPU_MEMORY_GB >= 24 else 8
skip_first = 16

print(f"Profile: {RUN_PROFILE}")
print(f"Fitting prompts: {len(fit_prompts)}")
print(f"Source layers: {source_layers}")
print(f"dim_batch: {dim_batch}")
print(f"max_seq_len: {max_seq_len}")

Profile: quick
Fitting prompts: 2
Source layers: [0, 4, 8, 12, 16, 20, 24, 26]
dim_batch: 8
max_seq_len: 64


## 8. Fit or reload the Jacobian lens

The checkpoint allows a stopped Kaggle session to resume from the last completed prompt when the same working directory is retained.

In [8]:
OUTPUT_DIR = Path("/kaggle/working/sarvam1_jlens")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

profile_tag = (
    f"{RUN_PROFILE}_p{len(fit_prompts)}"
    f"_s{layer_step}_d{dim_batch}_t{max_seq_len}"
)
CHECKPOINT_PATH = OUTPUT_DIR / f"{profile_tag}.fit_checkpoint.pt"
LENS_PATH = OUTPUT_DIR / f"{profile_tag}.lens.pt"

gc.collect()
torch.cuda.empty_cache()

if LENS_PATH.exists():
    lens = jlens.JacobianLens.load(str(LENS_PATH))
    print(f"Loaded lens: {LENS_PATH}")
else:
    lens = jlens.fit(
        model,
        prompts=fit_prompts,
        source_layers=source_layers,
        target_layer=None,
        dim_batch=dim_batch,
        max_seq_len=max_seq_len,
        skip_first=skip_first,
        checkpoint_path=str(CHECKPOINT_PATH),
        checkpoint_every=1 if RUN_PROFILE == "quick" else 2,
        resume=True,
    )
    lens.save(str(LENS_PATH))
    print(f"Saved lens: {LENS_PATH}")

print(lens)

Loaded lens: /kaggle/working/sarvam1_jlens/quick_p2_s4_d8_t64.lens.pt
JacobianLens(d_model=2048, n_prompts=2, source_layers=[0..26] (8 layers))


## 9. Compare Jacobian Lens and vanilla Logit Lens readouts

In [9]:
def format_top_tokens(logits: torch.Tensor, k: int = 8) -> str:
    values, token_ids = logits.topk(k)
    pieces = []
    for score, token_id in zip(values.tolist(), token_ids.tolist()):
        token = tokenizer.decode(
            [token_id],
            clean_up_tokenization_spaces=False,
        )
        pieces.append(f"{token!r} ({score:.2f})")
    return " | ".join(pieces)


jacobian_logits, final_logits, _ = lens.apply(
    model,
    PROMPT,
    layers=lens.source_layers,
    positions=[-1],
    max_seq_len=160,
    use_jacobian=True,
)

vanilla_logits, _, _ = lens.apply(
    model,
    PROMPT,
    layers=lens.source_layers,
    positions=[-1],
    max_seq_len=160,
    use_jacobian=False,
)

rows = []
for layer in lens.source_layers:
    rows.append(
        {
            "layer": layer,
            "Jacobian Lens": format_top_tokens(jacobian_logits[layer][0]),
            "Vanilla Logit Lens": format_top_tokens(vanilla_logits[layer][0]),
        }
    )

rows.append(
    {
        "layer": "final",
        "Jacobian Lens": format_top_tokens(final_logits[0]),
        "Vanilla Logit Lens": format_top_tokens(final_logits[0]),
    }
)

display(pd.DataFrame(rows))

,layer,Jacobian Lens,Vanilla Logit Lens
0,0,'ियनशिप' (27.42) | 'राबरी' (26.33) | 'चुसेट्स'...,'EIF' (16.64) | 'BLE' (14.01) | 'ത്തിറ' (13.29...
1,4,"'//--------' (22.56) | ""।''"" (21.08) | 'शुरुआ'...",'EIF' (19.31) | 'ંશી' (13.54) | '�' (13.16) | ...
2,8,"'शुरुआ' (23.17) | ""।''"" (21.45) | '//--------'...",'EIF' (13.41) | 'CAP' (13.04) | '/' (12.20) | ...
3,12,"'शुरुआ' (32.00) | ""।''"" (29.44) | '//--------'...",'साक्षा' (14.55) | 'జైన్' (13.12) | 'ets' (12....
4,16,"'शुरुआ' (30.25) | '//--------' (28.45) | ""।''""...",'EIF' (16.16) | 'साब' (16.05) | 'ूब' (14.29) |...
5,20,"'शुरुआ' (39.69) | ""।''"" (33.28) | 'प्रतिनि' (3...",'necessary' (17.95) | 'necess' (17.61) | 'need...
6,24,'आवश्यक' (17.72) | 'आवश्यकता' (14.59) | 'প্রয়...,'आवश्यक' (22.56) | 'necessary' (19.56) | 'જરૂર...
7,26,'आवश्यक' (6.91) | 'पाच' (5.06) | 'गर' (5.06) |...,'आवश्यक' (11.46) | 'आवश्यकता' (9.72) | 'necess...
8,final,'आवश्यक' (13.23) | 'आवश्यकता' (10.78) | 'प' (1...,'आवश्यक' (13.23) | 'आवश्यकता' (10.78) | 'प' (1...


## 10. Track smoke-related and recipe-related token ranks

Indic words can contain multiple tokenizer pieces. This cell records the first predicted piece for each probe and prints the exact decoded piece used in the rank calculation.

In [10]:
PROBE_WORDS = [
    "धूम",
    "अग्नि",
    "दाल",
    "जल",
    "मसाला",
    "चणक",
    "सामग्री",
    "पाकविधि",
]


def first_piece_id(word: str) -> int:
    token_ids = tokenizer(
        " " + word,
        add_special_tokens=False,
    ).input_ids
    if not token_ids:
        raise ValueError(f"No token IDs found for {word!r}")
    return int(token_ids[0])


probe_ids = {word: first_piece_id(word) for word in PROBE_WORDS}

probe_metadata = pd.DataFrame(
    [
        {
            "probe": word,
            "token_id": token_id,
            "decoded_piece": tokenizer.decode(
                [token_id],
                clean_up_tokenization_spaces=False,
            ),
        }
        for word, token_id in probe_ids.items()
    ]
)
display(probe_metadata)


def token_rank(logits: torch.Tensor, token_id: int) -> int:
    score = logits[token_id]
    return int((logits > score).sum().item()) + 1


def collect_probe_ranks(prompt: str, prompt_name: str) -> pd.DataFrame:
    layer_logits, model_logits, _ = lens.apply(
        model,
        prompt,
        layers=lens.source_layers,
        positions=[-1],
        max_seq_len=160,
        use_jacobian=True,
    )

    records = []
    for layer in lens.source_layers:
        vector = layer_logits[layer][0]
        record = {"prompt": prompt_name, "layer": layer}
        for word, token_id in probe_ids.items():
            record[word] = token_rank(vector, token_id)
        records.append(record)

    final_record = {"prompt": prompt_name, "layer": "final"}
    for word, token_id in probe_ids.items():
        final_record[word] = token_rank(model_logits[0], token_id)
    records.append(final_record)

    return pd.DataFrame(records)


rank_comparison = pd.concat(
    [
        collect_probe_ranks(PROMPT, "with distractor"),
        collect_probe_ranks(CONTROL_PROMPT, "control"),
    ],
    ignore_index=True,
)

display(rank_comparison)

,probe,token_id,decoded_piece
0,धूम,46296,धूम
1,अग्नि,52021,अग्नि
2,दाल,59131,दाल
3,जल,12229,जल
4,मसाला,4372,म
5,चणक,4622,च
6,सामग्री,14636,सामग्री
7,पाकविधि,54281,पाक


,prompt,layer,धूम,अग्नि,दाल,जल,मसाला,चणक,सामग्री,पाकविधि
0,with distractor,0,24392,40439,48970,66413,44213,46780,53505,44933
1,with distractor,4,5293,18876,6102,16118,2463,15822,25658,2392
2,with distractor,8,8476,13641,6586,13656,5176,13720,32934,3854
3,with distractor,12,14626,20071,14498,27825,16828,18473,47374,6793
4,with distractor,16,10939,7317,11811,10668,13112,11744,23282,2768
5,with distractor,20,14611,9457,4241,1265,19275,13573,13690,1207
6,with distractor,24,759,403,83,120,5151,819,268,74
7,with distractor,26,1014,255,135,114,699,314,206,63
8,with distractor,final,189,88,161,15,65,75,41,22
9,control,0,24537,40052,49214,66416,44421,46786,53724,45176


## 11. Build the interactive position-by-layer view

Click a cell to inspect the vocabulary ranking at a prompt position and layer. The pinned tokens include smoke-related and recipe-related Sanskrit pieces.

In [11]:
def truncate_to_tokens(text: str, max_tokens: int) -> str:
    token_ids = tokenizer(
        text,
        add_special_tokens=False,
    ).input_ids[:max_tokens]
    return tokenizer.decode(
        token_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )


analysis_text = truncate_to_tokens(
    PROMPT + distractor_completion,
    max_tokens=150,
)

pinned_token_ids = set()
alt_token = {}

for label, token_id in probe_ids.items():
    pinned_token_ids.add(token_id)
    decoded = tokenizer.decode(
        [token_id],
        clean_up_tokenization_spaces=False,
    )
    alt_token[token_id] = f"{label}: {decoded}"

slice_data = compute_slice(
    model,
    lens,
    analysis_text,
    top_n=10,
    max_tracked=48,
    pinned_token_ids=pinned_token_ids,
    layer_stride=1,
    last_n_tokens=130,
    max_seq_len=160,
    mask_display=True,
)

page, raw_bytes, payload_bytes = build_page(
    slice_data,
    analysis_text,
    title="Sarvam-1 Sanskrit Ignore-Instruction Experiment",
    description=(
        "Does the smoke distractor remain visible while the model "
        "forms a Panchmel dal recipe continuation?"
    ),
    pinned_token_ids=pinned_token_ids,
    mode="embed",
    alt_token=alt_token,
)

HTML_PATH = OUTPUT_DIR / f"{profile_tag}.interactive.html"
HTML_PATH.write_text(page, encoding="utf-8")

print(f"Raw slice bytes: {raw_bytes:,}")
print(f"Embedded payload bytes: {payload_bytes:,}")
print(f"Saved interactive page: {HTML_PATH}")

display(notebook_iframe(page, height=760))

Raw slice bytes: 351,000
Embedded payload bytes: 193,948
Saved interactive page: /kaggle/working/sarvam1_jlens/quick_p2_s4_d8_t64.interactive.html


## 12. Save the generated responses and experiment metadata

In [12]:
RESULTS_PATH = OUTPUT_DIR / f"{profile_tag}.results.json"

results = {
    "model_id": MODEL_ID,
    "run_profile": RUN_PROFILE,
    "gpu": GPU_NAME,
    "dtype": str(DTYPE),
    "prompt": PROMPT,
    "control_prompt": CONTROL_PROMPT,
    "distractor_completion": distractor_completion,
    "control_completion": control_completion,
    "source_layers": lens.source_layers,
    "fit_prompt_count": lens.n_prompts,
    "lens_path": str(LENS_PATH),
    "interactive_html_path": str(HTML_PATH),
}

RESULTS_PATH.write_text(
    json.dumps(results, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(f"Saved results: {RESULTS_PATH}")
print(f"Saved lens: {LENS_PATH}")
print(f"Saved visualization: {HTML_PATH}")

Saved results: /kaggle/working/sarvam1_jlens/quick_p2_s4_d8_t64.results.json
Saved lens: /kaggle/working/sarvam1_jlens/quick_p2_s4_d8_t64.lens.pt
Saved visualization: /kaggle/working/sarvam1_jlens/quick_p2_s4_d8_t64.interactive.html


## Reading the result

Look for three signals:

1. **Generation behavior:** Does the completion remain focused on ingredients and cooking steps, or does it return to the hill and smoke?
2. **Layer transition:** At which fitted layers do recipe-related tokens rise while smoke-related tokens fall?
3. **Distractor versus control:** Are the final predictions similar even when intermediate layers briefly retain smoke-related vocabulary?

A quick-profile lens is useful for checking the code and forming an initial hypothesis. Use the research profile, then expand the fitting corpus, before making a stronger interpretability claim.